---
title: Data Preparation - Cloud Storage & Profiling Statistik Data (Aiven - KNIME)
---

## Pendahuluan

### Review Tugas 1

#### Mengapa Perlu Melakukan Review Tugas 1?

Pengerjaan Tugas 2 memiliki hubungan langsung (*continuity*) dan merupakan kelanjutan dari hasil yang telah diperoleh pada Tugas 1. Dalam metodologi standar penambangan data **CRISP-DM (*Cross-Industry Standard Process for Data Mining*)**, alur kerja data science bergerak secara bertahap dari pemahaman bisnis (*Business Understanding*) dan pemahaman data (*Data Understanding*) menuju tahap penyiapan data (*Data Preparation*). 

Data mentah kualitas udara yang telah dikumpulkan, dieksplorasi, dan diekspor pada Tugas 1 akan menjadi masukan (*input*) utama pada Tugas 2. Oleh karena itu, *review* terhadap Tugas 1 sangat esensial untuk:

1. Mengingat kembali konteks wilayah, periode waktu, serta parameter kualitas udara yang diamati.
2. Memahami karakteristik, struktur kolom, dan transformasi awal (*rolling mean*) yang telah diterapkan pada dataset.
3. Memastikan keselarasan tujuan sebelum data tersebut dimigrasikan ke infrastruktur *cloud database* (Aiven) dan dianalisis profil statistiknya menggunakan KNIME Analytics Platform.

#### Ringkasan Pengerjaan dan Hasil Tugas 1

Pada Tugas 1 yang berjudul **"Data Crawling & Data Understanding (OpenEO - Sentinel-5P)"**, telah diselesaikan beberapa tahapan utama sebagai berikut:

1. **Tahap *Business Understanding*:**
   - **Pemahaman Konsep AQI & Polutan**: Mempelajari indeks kualitas udara (*Air Quality Index*) dan 6 parameter gas atmosfer utama yang disediakan oleh instrumen TROPOMI pada satelit **Sentinel-5P *Level-2***, yaitu:
     - **CO** (*Carbon Monoxide* / Karbon Monoksida)
     - **HCHO** (*Formaldehyde* / Formaldehida)
     - **NO2** (*Nitrogen Dioxide* / Nitrogen Dioksida)
     - **O3** (*Ozone* / Ozon)
     - **SO2** (*Sulphur Dioxide* / Sulfur Dioksida)
     - **CH4** (*Methane* / Metana)
   - **Penetapan Sasaran Pengamatan**: Menetapkan tujuan pemantauan tren kualitas udara di area tempat tinggal penulis (wilayah Surabaya–Sidoarjo) selama rentang waktu 1 tahun (24 Agustus 2025 hingga 24 Agustus 2026).

2. **Tahap *Data Understanding* & *Data Crawling*:**
   - **Autentikasi & Koneksi OpenEO**: Terhubung ke backend platform satelit *Copernicus OpenEO Data Space Ecosystem* (`openeo.dataspace.copernicus.eu`) menggunakan standar *OpenID Connect* (OIDC).
   - **Konfigurasi Wilayah (AOI)**: Menentukan poligon geografis *Area of Interest* (AOI) dalam format GeoJSON Polygon dan *spatial extent* (batas koordinat lintang dan bujur) tempat tinggal penulis.
   - **Pemrosesan Data Cube di Cloud**: Memuat koleksi data `SENTINEL_5P_L2` untuk keenam band polutan secara perulangan, menerapkan agregasi temporal harian (`aggregate_temporal_period`), menggabungkan kubus data (`merge_cubes`), dan melakukan agregasi spasial rata-rata terhadap geometri AOI (`aggregate_spatial`).
   - **Eksekusi Batch Job & Unduh NetCDF**: Menjalankan *batch job* pemrosesan di cloud OpenEO dan mengunduh berkas hasil akhir dalam format NetCDF (`kualitas_udara_sentinel5p.nc`).

3. **Eksplorasi Data & Prapemrosesan Awal:**
   - **Penghalusan Data (*Data Smoothing*)**: Menerapkan teknik rata-rata bergerak (*rolling mean* 30 hari) pada dataset `xarray` untuk mereduksi fluktuasi tajam harian (*noise*) akibat variasi cuaca atau tutupan awan.
   - **Pembersihan & Ekspor Data Tabular**: Menghapus fitur indeks dan koordinat statis yang redundan (`feature`, `lat`, `lon`, `feature_names`), menyelaraskan nama kolom indeks waktu menjadi `date`, dan mengekspor dataset ke dalam berkas `data_kualitas_udara.csv` dengan struktur kolom: `date`, `CO`, `HCHO`, `NO2`, `O3`, `SO2`, dan `CH4`.
   - **Visualisasi Deret Waktu & Spasial**: Membuat grafik kartesius (*line plot*) deret waktu untuk masing-masing polutan serta memvisualisasikan sebaran spasial polusi menggunakan peta interaktif *heatmap* berbasis *library* `folium`.

### Aiven


#### Apa itu Aiven?

[Aiven](https://aiven.io) adalah platform *Data Cloud Management* (*Data Platform as a Service*) yang menyediakan dan mengelola berbagai layanan basis data serta infrastruktur data *open-source* populer (seperti PostgreSQL, MySQL, Apache Kafka, OpenSearch, ClickHouse, dan Redis) secara terkelola penuh (*fully managed*). Aiven memungkinkan pengguna untuk meluncurkan database cloud di berbagai penyedia infrastruktur *cloud provider* terkemuka dunia, seperti Google Cloud Platform (GCP), Amazon Web Services (AWS), dan Microsoft Azure.

#### Fungsi dan Manfaat Aiven

Beberapa fungsi dan keunggulan utama Aiven dalam proyek data sains:
1. **Penyedia Layanan Database Terkelola (*Fully Managed Database*)**: Mengotomatiskan proses *deployment*, pemeliharaan, pencadangan (*backup* otomatis), pembaruan sistem, hingga keamanan database tanpa perlu konfigurasi server fisik manual.
2. **Aksesibilitas dan Integrasi Cloud**: Memungkinkan database (misalnya PostgreSQL atau MySQL) diakses secara aman dari berbagai alat analitik, ETL tools, maupun bahasa pemrograman melalui koneksi internet publik dengan enkripsi SSL/TLS.
3. **Skalabilitas Mudah**: Memudahkan penyesuaian kapasitas komputasi (*CPU*, *RAM*) dan ruang penyimpanan (*storage*) sesuai kebutuhan beban kerja data.
4. **Efisiensi Kolaborasi Tim**: Menyediakan *dashboard* berbasis web yang intuitif untuk memantau status performa, log query, dan konfigurasi pengguna/kredensial database secara terpusat.

Untuk informasi lebih lanjut dan pembuatan akun/layanan database cloud, Anda dapat mengunjungi situs resmi [Aiven](https://aiven.io).

### KNIME


#### Apa itu KNIME?
[KNIME](https://knime.com) (*Konstanz Information Miner*) adalah platform analitik data, pelaporan, dan integrasi *open-source* berbasis grafis (*visual workflow/drag-and-drop*) yang dirancang untuk kebutuhan sains data, rekayasa data (*data engineering*), pembersihan data (*data preparation/ETL*), hingga pemodelan *machine learning*. KNIME memungkinkan praktisi data membangun alur kerja pemrosesan data ujung-ke-ujung (*end-to-end data pipeline*) secara modular tanpa harus menulis kode program dari awal (*low-code/no-code environment*).

#### Fungsi dan Manfaat KNIME
Beberapa fungsi utama KNIME dalam analisis dan pengolahan data:
1. **Pembersihan dan Transformasi Data (*Data Cleaning & ETL*)**: Menyediakan modul (*nodes*) lengkap untuk filtering baris/kolom, penanganan nilai kosong (*missing values*), manipulasi tipe data, normalisasi, agregasi, hingga *merging* dan *joining* tabel.
2. **Profiling Statistik Data (*Data Profiling*)**: Menganalisis distribusi data, statistik deskriptif (mean, median, standar deviasi, kuartil), matriks korelasi antar variabel, serta mendeteksi nilai ekstrim/outlier secara otomatis dan interaktif.
3. **Konektivitas Database yang Luas**: Terhubung dengan lancar ke berbagai sumber data, termasuk basis data cloud (melalui JDBC driver untuk PostgreSQL/MySQL di Aiven), file lokal (CSV, Excel, Parquet), hingga layanan cloud storage.
4. **Visualisasi dan Eksplorasi Data**: Menyediakan berbagai node visualisasi bawaan (seperti histogram, scatter plot, box plot, bar chart, dan line plot) untuk memahami pola persebaran atribut data.
5. **Dukungan Machine Learning & Ekstensibilitas**: Memiliki integrasi bawaan dengan library pemodelan prediktif serta dapat diperluas dengan skrip Python atau R jika diperlukan operasi kustom.

Untuk mengunduh dan mempelajari fitur-fitur KNIME Analytics Platform lebih lanjut, Anda dapat mengunjungi website resmi [KNIME](https://knime.com).

## Apa yang Akan Dilakukan Sekarang?

- Memindahkan data CSV yang dihasilkan dari tugas ke-1, ke Database Cloud (Aiven)
- Menyambungkan Aiven dengan DB Manager lokal
- Menampilkan statistika dari data yang telah diupload ke Aiven menggunakan software KNIME.
- Mengidentifikasi lalu menjelaskan properti-properti dari statistik yang dihasilkan oleh KNIME, seperti:
    - Column
    - Min
    - Max
    - Mean
    - Median
    - Std. Dev.
    - Dll
- Memberi contoh perhitungan dari properti-properti yang ada (yang memungkinkan untuk dihitung), seperti:
    - Mean
    - Median
    - Std. Dev.
    - Kurtosis
    - Skewness

## Proses Pemindahan Data CSV ke Database Cloud (Via Aiven)

## Proses Menampilkan Statistika (Via KNIME)

## Properti-Properti yang Ditampilkan di Statistik KNIME